**Q: Show reviews of our vending machines.**

If you choose, define a DataFrame (and save as a view) in order to produce a report on net sentiment on vending machines, like the one below:

In [ ]:
answer = (
    reviews 
    .join(transactions, ['txn', 'txn_item']) 
    .join(vending_machines, 'vm_id') 
    .withColumn('sentiment', col('sentiment')['sentiment'].cast(StringType())) 
    .withColumn('review_score', 
                when(col('sentiment') == 'NEGATIVE', lit(-1)) 
                .when(col('sentiment') == 'NEUTRAL', lit(0)) 
                .when(col('sentiment') == 'POSITIVE', lit(1)) 
                .otherwise(lit(0))) 
    .groupBy('vm_id', 'loc') 
    .agg(
        [count('*').alias('num_reviews') 
        ,sum('review_score').alias('net_sentiment')
        ]) 
    .withColumn('annotated_vm', concat(col('vm_id'), lit('-'), 
                                       col('loc'), lit('('), 
                                       col('num_reviews'), lit(')'))) 
    .select('annotated_vm', 'net_sentiment') 
    .orderBy(col('net_sentiment').desc())
)

rows = answer.count()
rows

In [ ]:
answer.show(rows)

Save the DataFrame as a view.

In [ ]:
answer.createOrReplaceView('public.vending_machine_sentiment_vw')
session.sql("""select * from public.vending_machine_sentiment_vw""").show(rows)

As with the display showing product sentiment, we can query the view above in Snowsight and quickly produce the chart below.

<img src="images/Reviews_by_Vending_Machine.png" alt="DFOperationsTransform" style="width:60%;display:block;margin-left:5%;" />
